In [1]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0,org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,org.apache.iceberg:iceberg-aws-bundle:1.10.0 pyspark-shell'
import psycopg2
import psycopg2.extras
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("validation")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [3]:
PG_CONFIG = {
    "host":     os.environ.get("PG_HOST",     "postgres"),
    "port":     int(os.environ.get("PG_PORT", 5432)),
    "dbname":   os.environ.get("PG_DB",       "sourcedb"),
    "user":     os.environ.get("PG_USER",     "cdc_user"),
    "password": os.environ.get("PG_PASSWORD", "cdc_pass"),
}

SILVER_CUSTOMERS = "lakehouse.cdc.silver_customers"
SILVER_DRIVERS   = "lakehouse.cdc.silver_drivers"

errors = []

In [4]:
def pg_connect():
    return psycopg2.connect(**PG_CONFIG)


def check_row_counts(spark, conn):
    cur = conn.cursor()
    for pg_table, silver_table in [
        ("customers", SILVER_CUSTOMERS),
        ("drivers",   SILVER_DRIVERS),
    ]:
        cur.execute(f"SELECT COUNT(*) FROM {pg_table}")
        pg_count     = cur.fetchone()[0]
        silver_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {silver_table}").collect()[0]["cnt"]
        status       = "✓ PASS" if pg_count == silver_count else "✗ FAIL"
        print(f"[validation] {pg_table}: PostgreSQL={pg_count}, Silver={silver_count} → {status}")
        if pg_count != silver_count:
            errors.append(f"Row count mismatch for {pg_table}: PG={pg_count}, Silver={silver_count}")
    cur.close()

In [5]:
def check_spot_rows(spark, conn):
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
    cur.execute("SELECT id, name, email, country FROM customers ORDER BY RANDOM() LIMIT 3")
    for row in cur.fetchall():
        result = spark.sql(f"""
            SELECT id, name, email, country
            FROM {SILVER_CUSTOMERS}
            WHERE id = {row['id']}
        """).collect()
        if not result:
            msg = f"Customer id={row['id']} ({row['name']}) missing from silver"
            print(f"[validation] ✗ FAIL — {msg}")
            errors.append(msg)
        elif result[0]["name"] != row["name"] or result[0]["email"] != row["email"]:
            msg = f"Customer id={row['id']} data mismatch: PG={dict(row)}, Silver={result[0].asDict()}"
            print(f"[validation] ✗ FAIL — {msg}")
            errors.append(msg)
        else:
            print(f"[validation] ✓ PASS — customer id={row['id']} ({row['name']}) matches silver")
    cur.close()

In [6]:
def check_deletes_propagated(spark, conn):
    cur = conn.cursor()
    cur.execute("SELECT id FROM customers")
    pg_ids     = {r[0] for r in cur.fetchall()}
    silver_ids = {r["id"] for r in spark.sql(f"SELECT id FROM {SILVER_CUSTOMERS}").collect()}
    cur.close()

    ghost_ids = silver_ids - pg_ids
    if ghost_ids:
        msg = f"Silver has {len(ghost_ids)} ghost row(s) not in PostgreSQL: {ghost_ids}"
        print(f"[validation] ✗ FAIL — {msg}")
        errors.append(msg)
    else:
        print(f"[validation] ✓ PASS — no ghost rows (all deletes propagated correctly)")

In [7]:
conn = pg_connect()

print("[validation] === Silver CDC validation ===")
check_row_counts(spark, conn)
check_spot_rows(spark, conn)
check_deletes_propagated(spark, conn)

conn.close()

if errors:
    print(f"\n[validation] FAILED — {len(errors)} error(s):")
    for e in errors:
        print(f"  - {e}")
    raise Exception(f"Validation failed with {len(errors)} error(s): {errors}")

print("\n[validation] All checks passed ✓")

[validation] === Silver CDC validation ===
[validation] customers: PostgreSQL=10, Silver=10 → ✓ PASS
[validation] drivers: PostgreSQL=8, Silver=8 → ✓ PASS
[validation] ✓ PASS — customer id=1 (Alice Mets) matches silver
[validation] ✓ PASS — customer id=8 (Hiro Tanaka) matches silver
[validation] ✓ PASS — customer id=7 (Grace Kim) matches silver
[validation] ✓ PASS — no ghost rows (all deletes propagated correctly)

[validation] All checks passed ✓
